# HR Attrition — Production Preprocessing for XGBoost
**Pipeline:** Protected Attribute Isolation → Ordinal/OHE Encoding → HuggingFace Sentiment → Stratified Split → Class Weight Calculation

Target: `Attrition_Target` (1 = left, 0 = stayed). Highly imbalanced; handled via `scale_pos_weight`.

## 0. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import pipeline
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Global Config ────────────────────────────────────────────────────────────
RANDOM_STATE    = 42
TEST_SIZE       = 0.20
TARGET_COL      = 'Attrition_Target'
PROTECTED_COLS  = ['Age', 'Gender', 'Marital_Status']

# ── Data Paths ───────────────────────────────────────────────────────────────
PATH_EMPLOYEE     = 'employee_data.csv'
PATH_COMPENSATION = 'compensation_data.csv'
PATH_SURVEY       = 'survey_results.csv'
PATH_ATTRITION    = 'attrition_data.csv'
PATH_MARKET       = 'market_benchmarks.csv'

# ── Encoding Config ───────────────────────────────────────────────────────────
# NOTE: Values match actual Education_Level entries in employee_data.csv.
# Associate added; keys corrected from 'Bachelors'/'Masters' → 'Bachelor'/'Master'
ORDINAL_MAPS = {
    'Education_Level': {
        'High School': 1,
        'Associate':   2,
        'Bachelor':    3,
        'Master':      4,
        'PhD':         5
    }
}

NOMINAL_COLS     = ['Department', 'Employment_Type', 'Work_Location']
TEXT_COL         = 'Feedback_Comments'
SENTIMENT_COL    = 'Feedback_Sentiment'
SENTIMENT_MODEL  = 'distilbert-base-uncased-finetuned-sst-2-english'
SENTIMENT_BATCH  = 32

print("Config loaded. Random state:", RANDOM_STATE)

/home/kav/anaconda3/envs/athena/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config loaded. Random state: 42


## 1. Load Data

In [7]:
# ── Load all 5 source files ──────────────────────────────────────────────────
employee     = pd.read_csv(PATH_EMPLOYEE)
compensation = pd.read_csv(PATH_COMPENSATION)
survey       = pd.read_csv(PATH_SURVEY)
attrition    = pd.read_csv(PATH_ATTRITION)
market       = pd.read_csv(PATH_MARKET)

# ── Encode target BEFORE merge ────────────────────────────────────────────────
# Attrition_Date has 8192 NaNs by design (stayers have no exit date) — drop it.
attrition[TARGET_COL] = (attrition['Attrition_Status'] == 'Yes').astype(int)
attrition = attrition[['Employee_ID', TARGET_COL]]

# ── Derive Compa_Ratio from market benchmarks ─────────────────────────────────
# Compa_Ratio = employee Base_Salary / external Benchmark_Salary for same Role+Location.
# This is computed BEFORE merging into the master frame so the join is transparent.
# market.csv joins on Role + Location; employee.csv uses Work_Location — aliased below.
comp_with_benchmark = compensation.merge(
    employee[['Employee_ID', 'Role', 'Work_Location']],
    on='Employee_ID'
).merge(
    market.rename(columns={'Location': 'Work_Location'}),
    on=['Role', 'Work_Location'],
    how='left'
)
comp_with_benchmark['Compa_Ratio'] = (
    comp_with_benchmark['Base_Salary'] / comp_with_benchmark['Benchmark_Salary']
)

# ── Master merge (all joins on Employee_ID) ───────────────────────────────────
df = (
    employee
    .merge(survey[['Employee_ID', 'Job_Satisfaction', 'Work_Life_Balance',
                   'Management_Support', 'Career_Development',
                   'Engagement_Level', 'Feedback_Comments']],
           on='Employee_ID')
    .merge(comp_with_benchmark[['Employee_ID', 'Base_Salary', 'Bonus',
                                 'Stock_Options', 'Total_Compensation',
                                 'Compa_Ratio']],
           on='Employee_ID')
    .merge(attrition, on='Employee_ID')
)

df = df.drop(columns=['Employee_ID', 'Role'])

print(f"Master dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nTarget distribution:")
print(df[TARGET_COL].value_counts().to_string())
print(f"\nAttrition rate: {df[TARGET_COL].mean():.2%}")
print(f"\nNull check:\n{df.isnull().sum()[df.isnull().sum() > 0].to_string() or 'No nulls'}")

Master dataset: 10,000 rows × 20 columns

Target distribution:
Attrition_Target
0    8192
1    1808

Attrition rate: 18.08%

Null check:
Series([], )


## 2. Isolate Protected Attributes

**Why this matters mathematically:**  
XGBoost is a greedy ensemble of CART trees. At each node, it evaluates every feature using the gain formula:

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

where $G$ = sum of first-order loss gradients and $H$ = sum of second-order (Hessian) terms. If `Gender` or `Age` statistically correlates with attrition (even spuriously from historical HR decisions), this gain will be nonzero and the feature **will** be selected. The model encodes the bias into tree weights. Isolating them prevents this entirely while preserving the data for post-hoc fairness audits (e.g., equal FPR/FNR across demographic groups).

In [8]:
# Validate all protected columns exist before splitting
missing_protected = [c for c in PROTECTED_COLS if c not in df.columns]
if missing_protected:
    raise ValueError(f"Protected columns not found in dataframe: {missing_protected}")

# Preserve index alignment — critical for joining back after split
protected_df = df[PROTECTED_COLS].copy()
model_df     = df.drop(columns=PROTECTED_COLS).copy()

print(f"Protected attributes isolated: {PROTECTED_COLS}")
print(f"Modeling dataframe: {model_df.shape[1]} columns remain")

Protected attributes isolated: ['Age', 'Gender', 'Marital_Status']
Modeling dataframe: 17 columns remain


## 3. NLP Sentiment Extraction via HuggingFace

**Why this matters mathematically:**  
XGBoost splits on scalar thresholds like $x_j < \theta$. A raw string is a non-metric object with no defined ordering — it is impossible to compute a gradient or Hessian over it. A transformer encoder like DistilBERT maps the string into a dense continuous vector via its attention mechanism, then projects the pooled `[CLS]` embedding through a classification head to produce a probability $p \in [0, 1]$. We sign-flip negatives to yield a bipolar scalar $s \in [-1, 1]$, creating a proper real-valued feature the gradient boosting math can act on.

In [9]:
# ── Load the HuggingFace sentiment pipeline ───────────────────────────────────
# Uses DistilBERT fine-tuned on SST-2 (fast, accurate, ~250MB)
# device=0 uses GPU if available; device=-1 forces CPU
print(f"Loading sentiment model: {SENTIMENT_MODEL} ...")
sentiment_pipeline = pipeline(
    task            = "sentiment-analysis",
    model           = SENTIMENT_MODEL,
    truncation      = True,
    max_length      = 512,
    device          = -1,     # Change to 0 if you have a GPU
    batch_size      = SENTIMENT_BATCH
)

# ── Helper: map label+score → signed scalar ───────────────────────────────────
def score_to_signed(result: dict) -> float:
    """POSITIVE → [0, 1], NEGATIVE → [-1, 0]. NaN-safe."""
    s = result['score']  # always in (0, 1) — model confidence
    return s if result['label'] == 'POSITIVE' else -s

# ── Batch inference with progress bar ────────────────────────────────────────
# Fill NaN/empty text with a neutral placeholder before inference
texts = model_df[TEXT_COL].fillna('').str.strip().tolist()
texts_safe = [t if t else 'no comment provided' for t in texts]

print(f"Running sentiment inference on {len(texts_safe)} records (batch={SENTIMENT_BATCH}) ...")
raw_results = []
for i in tqdm(range(0, len(texts_safe), SENTIMENT_BATCH), desc="Sentiment batches"):
    batch = texts_safe[i : i + SENTIMENT_BATCH]
    raw_results.extend(sentiment_pipeline(batch))

model_df[SENTIMENT_COL] = [score_to_signed(r) for r in raw_results]
model_df = model_df.drop(columns=[TEXT_COL])

print(f"\nSentiment scores (signed, range [-1, 1]):")
print(model_df[SENTIMENT_COL].describe().round(4))

Loading sentiment model: distilbert-base-uncased-finetuned-sst-2-english ...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 14021.91it/s]


Running sentiment inference on 10000 records (batch=32) ...


Sentiment batches: 100%|██████████| 313/313 [00:35<00:00,  8.87it/s]


Sentiment scores (signed, range [-1, 1]):
count    10000.0000
mean         0.6660
std          0.7130
min         -0.9998
25%          0.9490
50%          0.9993
75%          0.9999
max          0.9999
Name: Feedback_Sentiment, dtype: float64


## 4. Categorical Encoding

**Why this matters mathematically:**

**Ordinal encoding** is appropriate when the variable has a *known, meaningful rank order*. For `Education_Level`, assigning integers `{1,2,3,4}` allows the tree to learn thresholds like `Education_Level >= 3` (i.e., Masters or above). The integer gap is irrelevant to CART — only the rank matters for split selection.

**One-Hot Encoding** is appropriate for *nominal* variables with no intrinsic order. If we label-encode `Department` as `{Sales=1, IT=2, HR=3}`, the model falsely assumes `IT(2) > Sales(1)`, which is meaningless and injects false gradient information. OHE converts each category into a binary indicator column, and XGBoost can split on any of them independently.

**Note on `drop_first`:** In logistic regression, dropping one dummy variable prevents perfect multicollinearity. XGBoost (a tree ensemble) is *not* affected by multicollinearity — it's not solving a linear system. Dropping a category removes information and can create split ambiguity. We keep all dummies.

In [10]:
# ── 4.1 Ordinal Encoding ─────────────────────────────────────────────────────
for col, mapping in ORDINAL_MAPS.items():
    if col not in model_df.columns:
        print(f"Warning: ordinal column '{col}' not found, skipping.")
        continue

    unseen = set(model_df[col].dropna().unique()) - set(mapping.keys())
    if unseen:
        raise ValueError(
            f"Column '{col}' contains values not in ordinal map: {unseen}.\n"
            f"Add them to ORDINAL_MAPS['{col}'] before continuing."
        )

    model_df[col] = model_df[col].map(mapping)
    print(f"  Ordinal encoded '{col}': {mapping}")

# ── 4.2 One-Hot Encoding ──────────────────────────────────────────────────────
# NOMINAL_COLS now includes Work_Location (9 cities + Remote — no natural rank).
# drop_first=False: XGBoost is not a linear model; keeping all dummies is correct.
existing_nominal = [c for c in NOMINAL_COLS if c in model_df.columns]
missing_nominal  = [c for c in NOMINAL_COLS if c not in model_df.columns]
if missing_nominal:
    print(f"Warning: nominal columns not found, skipping: {missing_nominal}")

model_df = pd.get_dummies(
    model_df,
    columns    = existing_nominal,
    drop_first = False,
    dtype      = np.int8
)

print(f"\nEncoding complete. Final shape: {model_df.shape}")
print(f"All columns:\n{list(model_df.columns)}")

  Ordinal encoded 'Education_Level': {'High School': 1, 'Associate': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}

Encoding complete. Final shape: (10000, 34)
All columns:
['Tenure_Years', 'Education_Level', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation', 'Compa_Ratio', 'Attrition_Target', 'Feedback_Sentiment', 'Department_Customer Support', 'Department_Engineering', 'Department_Finance', 'Department_HR', 'Department_Marketing', 'Department_Operations', 'Department_Product', 'Department_Sales', 'Employment_Type_Contract', 'Employment_Type_Full-Time', 'Employment_Type_Part-Time', 'Work_Location_Atlanta', 'Work_Location_Austin', 'Work_Location_Boston', 'Work_Location_Chicago', 'Work_Location_Denver', 'Work_Location_New York', 'Work_Location_Remote', 'Work_Location_San Francisco', 'Work_Location_Seattle']


## 5. Dtype Audit

XGBoost's C++ core requires all input features to be **float32** (or float64). Boolean or object columns cause silent conversion errors or raise exceptions at `.fit()` time. We audit and cast here — before the split — so problems surface early.

In [11]:
feature_cols = [c for c in model_df.columns if c != TARGET_COL]

# Check for any remaining object columns (would cause XGBoost to crash)
object_cols = model_df[feature_cols].select_dtypes(include='object').columns.tolist()
if object_cols:
    raise ValueError(
        f"Object-type columns remain after encoding — XGBoost cannot ingest these: {object_cols}\n"
        "Add them to NOMINAL_COLS or ORDINAL_MAPS."
    )

# Cast everything to float32 for XGBoost compatibility
model_df[feature_cols] = model_df[feature_cols].astype(np.float32)

print("Dtype audit passed. All feature columns cast to float32.")
print(model_df[feature_cols].dtypes.value_counts())

Dtype audit passed. All feature columns cast to float32.
float32    33
Name: count, dtype: int64


## 6. Stratified Train/Test Split

**Why stratification is mathematically necessary:**  
With a class imbalance of e.g. 85/15, a naive random 80/20 split has variance in the minority class count. In a small test set, you could end up with 0 positive samples by chance — AUC, F1 and Precision-Recall curves become undefined or misleading. The `stratify=y` argument uses `StratifiedShuffleSplit` internally, which enforces that the ratio $P(Y=1)$ is identical in both train and test — making evaluation metrics reliable.

**Note on splitting the protected dataframe:** `train_test_split` only splits paired arrays if passed together. We use index alignment to correctly slice `protected_df` *after* the main split — no data leakage, no index mismatch.

In [12]:
X = model_df.drop(columns=[TARGET_COL])
y = model_df[TARGET_COL]

# ── Stratified split on X and y ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y    # Enforces identical class ratio in both sets
)

# ── Use index alignment to split protected_df — avoids train_test_split bugs ──
# train_test_split does NOT support splitting >2 arrays reliably across versions.
# Index-based slicing is safer and more explicit.
protected_train = protected_df.loc[X_train.index]
protected_test  = protected_df.loc[X_test.index]

# ── Verify index alignment ────────────────────────────────────────────────────
assert (X_train.index == protected_train.index).all(), "Index mismatch: X_train vs protected_train"
assert (X_test.index  == protected_test.index).all(),  "Index mismatch: X_test vs protected_test"

print(f"Split sizes:")
print(f"  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"\nClass distribution (attrition rate):")
print(f"  Full dataset : {y.mean():.4f} ({y.sum()}/{len(y)})")
print(f"  Train set    : {y_train.mean():.4f} ({y_train.sum()}/{len(y_train)})")
print(f"  Test set     : {y_test.mean():.4f} ({y_test.sum()}/{len(y_test)})")

Split sizes:
  X_train: (8000, 33)  |  X_test: (2000, 33)

Class distribution (attrition rate):
  Full dataset : 0.1808 (1808/10000)
  Train set    : 0.1807 (1446/8000)
  Test set     : 0.1810 (362/2000)


## 7. Calculate `scale_pos_weight`

**Why this is mathematically necessary:**  
XGBoost minimizes the regularized objective:

$$\mathcal{L} = \sum_i l(y_i, \hat{y}_i) + \sum_k \Omega(f_k)$$

where $l$ is log-loss. With 85% negatives, the gradient signal from the minority class is 5.7× weaker. The optimizer converges toward predicting 0 everywhere — achieving high accuracy but zero recall. `scale_pos_weight` scales the loss gradient **and Hessian** for positive instances by a factor $w$:

$$g_i^{\text{scaled}} = w \cdot g_i, \quad h_i^{\text{scaled}} = w \cdot h_i \quad \text{when } y_i = 1$$

The recommended value is $w = N_{\text{neg}} / N_{\text{pos}}$, which restores gradient balance. Computed on **train set only** to prevent test set leakage.

In [13]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

if n_pos == 0:
    raise ValueError("No positive samples in training set — check your data or stratification.")

scale_pos_weight = n_neg / n_pos

print("═" * 55)
print(f"  Training negatives (stayed) : {n_neg}")
print(f"  Training positives (left)   : {n_pos}")
print(f"  scale_pos_weight            : {scale_pos_weight:.4f}")
print("═" * 55)
print()
print("Pass this into your XGBoost model:")
print(f"  XGBClassifier(scale_pos_weight={scale_pos_weight:.4f}, ...)")
print()
print("Or in Optuna, expose it as a fixed parameter:")
print(f"  params['scale_pos_weight'] = {scale_pos_weight:.4f}")

═══════════════════════════════════════════════════════
  Training negatives (stayed) : 6554
  Training positives (left)   : 1446
  scale_pos_weight            : 4.5325
═══════════════════════════════════════════════════════

Pass this into your XGBoost model:
  XGBClassifier(scale_pos_weight=4.5325, ...)

Or in Optuna, expose it as a fixed parameter:
  params['scale_pos_weight'] = 4.5325


## 8. Final Outputs Summary

All arrays are ready to pass into XGBoost or Optuna.

In [14]:
print("━" * 55)
print("READY-TO-USE OBJECTS")
print("━" * 55)
print(f"  X_train          : {X_train.shape}  (float32, no protected attrs, no text)")
print(f"  X_test           : {X_test.shape}")
print(f"  y_train          : {y_train.shape}  (int, stratified)")
print(f"  y_test           : {y_test.shape}")
print(f"  protected_train  : {protected_train.shape}  (for fairness audits)")
print(f"  protected_test   : {protected_test.shape}")
print(f"  scale_pos_weight : {scale_pos_weight:.4f}")
print("━" * 55)
print()
print("Quickstart for Optuna tuning:")
print("""
import xgboost as xgb
import optuna

def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 9),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # Fixed — do NOT tune this:
        'scale_pos_weight': scale_pos_weight,
        'eval_metric':      'aucpr',  # Better than AUC-ROC for imbalanced data
        'use_label_encoder': False,
        'random_state':     42
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              early_stopping_rounds=50,
              verbose=False)
    preds = model.predict_proba(X_test)[:, 1]
    from sklearn.metrics import average_precision_score
    return average_precision_score(y_test, preds)  # Maximize PR-AUC

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)
""")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
READY-TO-USE OBJECTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  X_train          : (8000, 33)  (float32, no protected attrs, no text)
  X_test           : (2000, 33)
  y_train          : (8000,)  (int, stratified)
  y_test           : (2000,)
  protected_train  : (8000, 3)  (for fairness audits)
  protected_test   : (2000, 3)
  scale_pos_weight : 4.5325
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Quickstart for Optuna tuning:

import xgboost as xgb
import optuna

def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 9),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial